# Tutorial A: a screw swimming along a tube

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Microrobotics-Simulation-Framework/MIME/blob/v0.3-colab-notebooks/docs/notebooks/tutorial_a.ipynb)

De Jongh's FL-9 screw rests on the floor of a quarter-inch tube of water. A uniform field of 1.2 mT turns about the
tube's axis at 10 Hz, the screw turns with it, and it swims along the tube. This is the reproduction arm of
`examples/script_a.py`, the experiment of the user guide's tutorial A (`docs/guide/tutorial_a.md`): every value is
declared with where it comes from, the earlier (0.2) line's run of A. Its drag is a surrogate whose weights ship
with MIME, so nothing is fetched. The notebook installs MIME 0.3.0.dev1 from PyPI, builds the arm, runs it for 5 simulated
seconds with MIME's live display, plots the swim speed and the screw's position along the tube, and shows the run in
the browser view. It needs no GPU.

In [ ]:
!pip install -q --pre "mime-engine==0.3.0.dev1"

## The experiment

The cells tagged `example` are the example's own code, copied from `examples/script_a.py` unchanged; MIME's test suite
(`tests/examples/test_notebooks.py`) fails when they differ from it. The imports are the example's, less those only its
record arm uses. Edit the cells here to try a change; the field's frequency and the step are `script_a()`'s
arguments, and the frequency is set where the arm is built, below.

In [ ]:
import numpy as np

from mime import Apparatus, Body, Experiment, Medium, State, Structure
from mime.data import asset
from mime.dynamics import RigidInertial
from mime.fields import UniformRotating
from mime.interactions import KEPT, Drag, Gravity, Lubrication, Magnetic
from mime.observations import SwimSpeed
from mime.scene import Scene, Screw, Tube, X, Z
from mime.things import Part
from mime.units import (
    A_m2,
    Hz,
    dimensionless,
    kg,
    kg_m2,
    kg_per_m3,
    m,
    m3,
    m_per_s,
    m_per_s2,
    mm,
    mm_per_s,
    mPa_s,
    ms,
    mT,
    rad_per_s,
    s,
)

In [ ]:
REFERENCE = "the 0.2 line's script A (MIME-port041 at 6485516)"
#: The reference held its body by its centre, its extent kept from the wall (the contact 0.1.0, which MIME keeps
#: registered for reproductions like this one); a new experiment writes Contact.Projection(...), which holds the body's
#: hull, its orientation read.
CentreContact, _ = KEPT["contact", "0.1.0"]
#: FL-9: R 1.56 mm, L 7.47 mm, nu 2.33, modulation 0.33, two starts (de Jongh et al. 2025, Table 1).
FL9 = Screw(radius=mm(1.56), length=mm(7.47), wavenumber=dimensionless(2.33), modulation=dimensionless(0.33),
            starts=2)

`script_a()` declares the world (up is +z, so gravity points along -z), the tube along x and the water in it, the screw
on the tube's floor with its declared mass, inertia, buoyant volume and magnetic moment, and the rotating field. Then
it attaches what acts on the screw: the drag surrogate with lubrication, the field's torque on the screw's moment,
gravity with buoyancy, and the contact that keeps it in the tube. It observes the swim speed along the tube's axis.

In [ ]:
def script_a(frequency=Hz(10.0), dt=ms(0.5)) -> Experiment:
    # region: world
    # The world: up is +z, so gravity points along -z.
    experiment = Experiment("script_a", scene=Scene.new(up=Z, units="m"), gravity=m_per_s2(9.81), dt=dt)

    # The 1/4" silicone tube, along +x, and the water inside it.
    tube = experiment.add_structure(Structure("tube", geometry=Tube(axis=+X, inner_radius=mm(3.175), wall=mm(1.6),
                                                                    length=mm(100.0))))
    water = experiment.add_medium(Medium("water", fills=tube.lumen))
    experiment.declare("water.density", kg_per_m3(1000.0), reason=REFERENCE)
    experiment.declare("water.viscosity", mPa_s(1.0), reason=REFERENCE)
    # endregion world

    # region: body
    # FL-9, resting on the tube's floor 1 mm below the axis, its own axis along the tube's. Its shell is the de Jongh
    # helix (de Jongh et al. 2025, Eq. 1 and Table 1), which gives the drag its shape and the body its extent.
    umr = experiment.add_body(Body("umr", dynamics=RigidInertial(), initial=State(
        position=mm((0.0, 0.0, -1.0)), orientation=(1.0, 0.0, 0.0, 0.0), velocity=m_per_s((0.0, 0.0, 0.0)),
        angular_velocity=rad_per_s((0.0, 0.0, 0.0))), parts={"shell": Part(FL9)}))
    experiment.declare("umr.mass", kg(8.491116020725203e-05),
                       reason=f"{REFERENCE}: its m_eff, (410 + 1000) kg/m^3 times FL-9's volume, with no added mass")
    experiment.declare("umr.inertia", kg_m2((1.0331989974018431e-10,) * 3),
                       reason=f"{REFERENCE}: its I_eff, m R^2 / 2 for a solid cylinder of FL-9's radius, used about "
                              f"every axis alike")
    experiment.declare("umr.buoyant_volume", m3(6.022068099805108e-08), reason=f"{REFERENCE}: FL-9's volume")
    experiment.declare("umr.centre_of_buoyancy", m((0.0, 0.0, 0.0)),
                       reason="the reference's body has one density, so its centre of buoyancy is its centre of mass")
    experiment.declare("umr.magnetic_moment", A_m2((0.0, 2 * 8.4e-4, 0.0)),
                       reason=f"{REFERENCE}: two magnets of 8.4e-4 A m^2, across the body's axis")
    # endregion body

    # region: field
    # The field, starting along the body's moment and turning right-handed about the tube's axis.
    field = experiment.add_apparatus(Apparatus("field", fields={
        "magnetic": UniformRotating(rotation_axis=tube.axis, initial_direction=(0.0, 1.0, 0.0))},
        commands={"frequency": frequency, "amplitude": mT(1.2)}))
    # endregion field

    # region: laws
    experiment.attach(Drag.Surrogate(weights=asset("dejongh_mlp_v2"), lubrication=Lubrication(epsilon=mm(0.15))),
                      body=umr, medium=water)
    experiment.attach(Magnetic.Torque(), body=umr, field=field.magnetic)
    experiment.attach(Gravity(), body=umr, medium=water)
    experiment.attach(CentreContact.Projection(margin=mm(0.1)), body=umr, surface=tube.inner_surface)
    experiment.observe(SwimSpeed(umr, along=tube.axis))
    # endregion laws
    return experiment

## Building it

`.build()` turns the declarations into a graph and a build report, which lists every value an entry reads and where it
came from ([reading a build report](https://github.com/Microrobotics-Simulation-Framework/MIME/blob/v0.3.0.dev1/docs/guide/build_report.md)).
The build also warns, before anything runs, that the drive turns the screw far above the Reynolds number the drag
surrogate is declared for: see what this notebook claims, at the end.

In [ ]:
FREQUENCY = Hz(10.0)
built = script_a(frequency=FREQUENCY).build()
print(built.report.text())

## The run

Five simulated seconds, the length of the earlier line's reference run, in 10 000 steps of 0.5 ms. `run_to` writes the
run folder, records the swim speed at every step and the screw's pose every 10 ms (ten samples a turn of the field), and
shows MIME's live display: compiling, the run's bar, then the saving phase.

In [ ]:
DURATION, STAGE_EVERY = s(5.0), ms(10.0)
written = built.run_to("script_a_run", DURATION, record=["swim_speed:umr.speed"], stage_every=STAGE_EVERY)
print(written.result.text())

The example's reading: the swim speed along the tube over the second half of the run, the reference's window.

In [ ]:
speed = written.result["swim_speed:umr"].speed
half_s, end_s = DURATION.to(s) / 2, DURATION.to(s)
later = speed.between(s(half_s)).mean()
print(f"over {half_s:g} to {end_s:g} s: swim speed along the tube {later.to(mm_per_s):.6f} mm/s")

## The swim speed and the screw's position along the tube

Plotted with matplotlib, which Colab ships (it is not a dependency of MIME). While the screw spins up, the speed at
every step ripples with each turn of the field; its mean over each turn is drawn over it. The position is the screw's centre along the tube's
axis, read from the run's recording.

In [ ]:
import matplotlib.pyplot as plt
from pxr import Usd

time_s, speed_mm_s = speed.time.to(s), speed.values.to(mm_per_s)
per_turn = round(1.0 / (FREQUENCY.to(Hz) * speed.dt * speed.every))  # samples in one turn of the field
turns = len(speed_mm_s) // per_turn
turn_end_s = time_s[per_turn - 1::per_turn][:turns]
turn_mean = speed_mm_s[:turns * per_turn].reshape(turns, per_turn).mean(axis=1)

stage = Usd.Stage.Open(str(written.recording))
translate = stage.GetPrimAtPath("/World/umr").GetAttribute("xformOp:translate")
codes = translate.GetTimeSamples()
sample_s = np.array(codes) / stage.GetTimeCodesPerSecond()
along_mm = np.array([translate.Get(code)[0] for code in codes]) * 1e3

figure, (upper, lower) = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True, layout="constrained")
upper.plot(time_s, speed_mm_s, color="#2a78d6", linewidth=0.6, alpha=0.45, label="at every step")
upper.plot(turn_end_s, turn_mean, color="#eb6834", linewidth=2, label="mean over each turn of the field")
upper.axvspan(half_s, end_s, color="#898781", alpha=0.12, linewidth=0, label="the reference's window")
upper.set_ylabel("swim speed along the tube (mm/s)")
upper.set_title("Swim speed")
lower.plot(sample_s, along_mm, color="#2a78d6", linewidth=2)
lower.set_ylabel("position along the tube (mm)")
lower.set_xlabel("time (s)")
lower.set_title("The screw's centre along the tube's axis")
for axes in (upper, lower):
    axes.grid(color="#e1e0d9", linewidth=0.8)
    axes.spines[["top", "right"]].set_visible(False)
upper.legend(frameon=False, loc="lower right")
plt.show()

## The browser view

`mime.viz.display` shows the run's recording as a page inside the notebook: drag to turn the view, scroll to zoom
toward the pointer, and use the slider to move through the run. The viewer loads from jsDelivr, so the cell needs the
network; `offline=True` inlines it instead, adding about 1.4 MB to the notebook. GitHub's notebook preview runs no
script, so there the cell is blank.

In [ ]:
from mime import viz

viz.display("script_a_run", caption=(
    "Script A's reproduction arm, as the earlier line's run of A declared it. Its drag surrogate is declared up to a "
    "Reynolds number of 0.1, and this drive turns the screw far above it: the result is unvalidated there, which is "
    "not the same as measured to be wrong."))

## What this notebook claims, and what it does not

It claims that the reproduction arm builds and runs as written, and that its output is the output shown. It claims
nothing about the swim speed as a prediction of the flow or of the hardware.

- **The build's warning and the post-run watch.** From the field's declared frequency, the screw's diameter and water's
  density and viscosity, the build computes the turning Reynolds number at the drive's rate and finds it far above the
  0.1 the surrogate rung declares itself valid up to; after the run, the watch on the drag reads the run's own number
  and finds it above too. Both are flags, never refusals. The rung's range is declared, not fitted: outside it the
  result is unvalidated, which is not the same as measured to be wrong.
- **What MIME's tests hold about A.** `tests/differential/test_script_a.py` holds the reproduction arm to the frozen
  outputs of the earlier line's run of A, with every difference declared before the comparison first ran: it bounds
  how far MIME 0.3's A is from the earlier line's A at this operating point. The earlier code is the weakest oracle
  there is, so the agreement says nothing about the flow or about hardware. This notebook does not run that
  comparison.
- **The record arm**, which takes water's values and the magnets' moment from records instead of declaring them, is in
  the same example (`record_arm()`) and in the guide's tutorial A.